In [1]:
import pickle
import sys
import copy
import time

import cobra
import sympy


import multiprocessing
import multiprocessing.pool
# from multiprocessing import Process
# from threading import Thread

sys.path.insert(1, '/home/hratch/Projects/human_me/scripts/')
from utils import functions as func
from utils import parameters as params

In [2]:
# import pandas as pd
# build_files_path = '/data2/hratch/human_me/build_files/'
# human_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/toy_model.json')
# full_model = cobra.io.load_json_model('/data2/hratch/human_me/input_files/recon2_2.json')
# full_model_public = cobra.io.read_sbml_model(lp_path + 'recon2_2.xml')

# required_metabolites = pd.read_csv(build_files_path + 'required_metabolic_model_metabolites.csv', index_col = 0)

# me_model_og = copy.deepcopy(me_model)

In [3]:
from tqdm import tqdm

In [4]:
#For some reason, this protein complex used to catalyze the formation of the pre40s complex causes infeasiblity. 
#Including any of the 2 of the complex subcomponents in the precursors_fail list makes it work. Or their earlier 
#versions (up to unfolded_protein_c).

error_metabolites = ['pre40s_rrna_protein_COMPLEX_FORMATIONn_protein_complex[n]']
precurors_work = ['HGNC:21173_folded_protein[n]', 'HGNC:32790_folded_protein[n]']
precursors_fail = ['HGNC:25542_folded_protein[n]', 'HGNC:29100_folded_protein[n]']

# folded_protein[n] <-- folded_protein[c] <-- unfolded_protein[c] <-- 
# adding unfolded_protein[c] of precursors fail works too, don't need both of the precursors fail, just 1...
error_metabolites = precursors_fail.copy()

In [9]:
lp_path = '/data2/hratch/human_me/test_lp/'


with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
    me_model = pickle.load(handle)

In [5]:
def remove_metabolite(test_metabolites = [], mu_val = 0.01):
    
    with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
        me_model = pickle.load(handle)
    
    me_metabolites = [m.id for m in me_model.metabolites if 'deg_proxy' in m.id or ('mrna[n]' in m.id and 'premrna' not in m.id and 'lariats' not in m.id)]
    me_metabolites = []
    me_metabolites += error_metabolites

    for tm in test_metabolites:
        me_metabolites.remove(tm)
    
    ra = []
    for mm_id in me_metabolites: #me_metabolites:
        try:
            mm_obj = me_model.metabolites.get_by_id(mm_id)
        except:
            mm_obj = params.human_model.metabolites.get_by_id(mm_id)
        r = cobra.Reaction('TEST_' + mm_obj.id)
        r.add_metabolites({mm_obj: 1}, reversibly = True)
        ra.append(r)
    if len(ra) > 0:
        me_model.add_reactions(ra)
    sln, status, _ = me_model.solve_lp(mu_val = mu_val)
    return sln, status

In [6]:
#max 1e-4
sln,status = remove_metabolite(mu_val = 0)

Getting MINOS parameters...
Done in 3.11224 seconds with status 0


In [ ]:
#max 1e-4
sln,status = remove_metabolite(mu_val = 1e-9)

In [11]:
#max 1e-4
sln,status = remove_metabolite(mu_val = 1e-4)

Getting MINOS parameters...
Done in 230.049 seconds with status 0


In [12]:
#max 1e-4
sln,status = remove_metabolite(mu_val = 1e-3)

Getting MINOS parameters...
Done in 191.446 seconds with status 1


In [ ]:
# older version: mrna separate, c1b only incorporated 

In [31]:
sln,status = remove_metabolite(mu_val = 0.0001)

Getting MINOS parameters...
Done in 111.685 seconds with status 0


In [10]:
[r for r in me_model.reactions if 'DM_' in r.id and 'rna' in r.id]

[]

In [52]:
for r in [r for r in me_model.reactions if 'DM_' in r.id and 'rna' in r.id]:
    if sln[me_model.reactions.index(r.id)] > 0:
        print(r.id)

DM_other_rna_HGNC:10312
DM_other_rna_HGNC:10313
DM_other_rna_HGNC:10316
DM_other_rna_HGNC:10317
DM_other_rna_HGNC:10325
DM_other_rna_HGNC:10327
DM_other_rna_HGNC:17050
DM_other_rna_HGNC:10328
DM_other_rna_HGNC:10331
DM_other_rna_HGNC:10336
DM_other_rna_HGNC:10340
DM_other_rna_HGNC:10344
DM_other_rna_HGNC:10345
DM_other_rna_HGNC:10359
DM_other_rna_HGNC:10346
DM_other_rna_HGNC:10347
DM_other_rna_HGNC:10349
DM_other_rna_HGNC:10350
DM_other_rna_HGNC:10354
DM_other_rna_HGNC:12458
DM_other_rna_HGNC:10545
DM_other_rna_HGNC:10547
DM_mrna[n]_HGNC:10920
DM_other_rna_HGNC:10930
DM_other_rna_HGNC:11004
DM_other_rna_HGNC:11007
DM_other_rna_HGNC:11022
DM_other_rna_HGNC:11047
DM_other_rna_HGNC:11279
DM_other_rna_HGNC:11863
DM_other_rna_HGNC:12390
DM_mrna[n]_HGNC:12590
DM_other_rna_HGNC:12591
DM_premrna_HGNC:13213
DM_other_rna_HGNC:13812
DM_mrna[n]_HGNC:14937
DM_mrna[n]_HGNC:1516
DM_other_rna_HGNC:17174
DM_mrna[n]_HGNC:17194
DM_mrna[n]_HGNC:17325
DM_premrna_HGNC:17325
DM_other_rna_HGNC:1769
DM_mrna[n]

In [40]:
[r for r in me_model.reactions if 'TRANSLATION_ELONGATION' in r.id][1].reaction

'mu/(mu + 0.02) HGNC:10404_mrna[c] + 2.93213986783079e6*mu  5.86427973566157e8 TRANSLATION_ELONGATIONc_complex_protein_complex[c] + 31.62758551999832 biomass_tRNA + 25 charged_generic_A_trna[c] + 5 charged_generic_C_trna[c] + 12 charged_generic_D_trna[c] + 10 charged_generic_E_trna[c] + 11 charged_generic_F_trna[c] + 43 charged_generic_G_trna[c] + 4 charged_generic_H_trna[c] + 17 charged_generic_I_trna[c] + 24 charged_generic_K_trna[c] + 19 charged_generic_L_trna[c] + 7 charged_generic_M_trna[c] + 4 charged_generic_N_trna[c] + 16 charged_generic_P_trna[c] + 6 charged_generic_Q_trna[c] + 24 charged_generic_R_trna[c] + 14 charged_generic_S_trna[c] + 21 charged_generic_T_trna[c] + 21 charged_generic_V_trna[c] + 3 charged_generic_W_trna[c] + 7 charged_generic_Y_trna[c] + 293 gtp[c] + 294 h2o[c] --> HGNC:10404_unfolded_protein[c] + 31.324064600000053 biomass_protein + 293 gdp[c] + 293 generic_trna[c] + 586 h[c] + 293 pi[c]'

working version: only c1B coupling and separate demand reaction for each mrna metabolite type

In [38]:
# with open(lp_path + 'toy_me_model.pickle', 'rb') as handle:
#     me_model = pickle.load(handle)
# sln, status, _ = me_model.solve_lp(mu_val = 0.01)

In [ ]:
# mrna[n] requirement: no flux through transcription elongation and transcription processing!!
# mrna_deg_proxy requirement: no flux through transcription degradation reaction
# additionally, no flux through protein degradation (polyub reaction, deubiquitination, or proteosome) reaction; 
# 2 lariat degradation reactions created